# Notebook 2: Create Appointment table
Creates 100 completed appointments in the past 3 months with required distribution constraints.

In [ ]:
import random
from datetime import datetime, timedelta
from collections import Counter
from pyspark.sql import functions as F

SEED = 20260402
random.seed(SEED)

patients = [r.patientId for r in spark.table('patient').select('patientId').collect()]
providers = [r.providerId for r in spark.table('provider').select('providerId').collect()]

if len(patients) != 50:
    raise ValueError('Expected 50 patients from Notebook 1.')
if len(providers) != 20:
    raise ValueError('Expected 20 providers from Notebook 1.')

In [ ]:
now = datetime.now()
window_start = now - timedelta(days=90)
durations = [15, 30, 45, 60]
appt_types = ['Follow-up', 'Consultation', 'Annual Physical', 'Lab Review', 'Specialist Visit']

# Guarantee constraints:
# 1) Every patient has at least one appointment
# 2) Several patients have exactly 5 appointments
# 3) Total appointments = 100
heavy_patients = patients[:6]  # 6 patients x 5 appointments = 30
appt_patient_ids = []
for p in heavy_patients:
    appt_patient_ids.extend([p] * 5)

remaining_patients = patients[6:]
for p in remaining_patients:
    appt_patient_ids.append(p)  # at least one each -> +44 (total now 74)

# Fill the remaining 26 slots semi-randomly, excluding heavy patients so they remain at exactly 5.
for _ in range(26):
    appt_patient_ids.append(random.choice(remaining_patients))

if len(appt_patient_ids) != 100:
    raise ValueError('Appointment allocation did not reach 100 rows.')

random.shuffle(appt_patient_ids)

appointments = []
for i, patient_id in enumerate(appt_patient_ids, start=1):
    scheduled = window_start + timedelta(minutes=random.randint(0, 90 * 24 * 60 - 1))
    appointments.append({
        'appointmentId': f"APT{i:04d}",
        'patientId': patient_id,
        'providerId': random.choice(providers),
        'scheduledTime': scheduled.isoformat(timespec='seconds'),
        'duration': random.choice(durations),
        'type': random.choice(appt_types),
        'status': 'completed'
    })

appointments_df = spark.createDataFrame(appointments)
appointments_df = appointments_df.select(
    'appointmentId', 'patientId', 'providerId', 'scheduledTime', 'duration', 'type', 'status'
)
appointments_df = appointments_df.withColumn('scheduledTime', F.to_timestamp('scheduledTime'))
appointments_df.write.mode('overwrite').format('delta').saveAsTable('appointment')

display(spark.table('appointment').orderBy('appointmentId'))

In [ ]:
appt = spark.table('appointment')
patient_count = appt.select('patientId').distinct().count()
total_count = appt.count()
status_ok = appt.filter(F.col('status') != 'completed').count() == 0
window_ok = appt.filter((F.col('scheduledTime') < F.lit(window_start)) | (F.col('scheduledTime') > F.current_timestamp())).count() == 0

patient_freq = Counter([r.patientId for r in appt.select('patientId').collect()])
patients_with_5 = sum(1 for c in patient_freq.values() if c == 5)

print('appointment_count_100:', total_count == 100)
print('all_50_patients_have_appointments:', patient_count == 50)
print('all_completed:', status_ok)
print('window_past_3_months:', window_ok)
print('patients_with_exactly_5_appointments:', patients_with_5)